# Part A - Data Preparation
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  
**Course:** CS4241 - Introduction to Artificial Intelligence

In [13]:
import pandas as pd
import fitz
import json
import os

## 1. Load and Clean the CSV

In [14]:
df = pd.read_csv('Ghana_Election_Result.csv', encoding='utf-8-sig')
print(df.shape)
df.head()

(615, 8)


,Year,Old Region,New Region,Code,Candidate,Party,Votes,Votes(%)
0,2020,Brong Ahafo Region,Ahafo Region,NPP,Nana Akufo Addo,NPP,145584,55.04%
1,2020,Brong Ahafo Region,Ahafo Region,NDC,John Dramani Mahama,NDC,116485,44.04%
2,2020,Brong Ahafo Region,Ahafo Region,Others,Ivor Kobina Greenstreet,CPP,191,0.07%
3,2020,Brong Ahafo Region,Ahafo Region,Others,David Asibi Ayindenaba Apasera,PNC,83,0.03%
4,2020,Brong Ahafo Region,Ahafo Region,Others,Alfred Kwame Asiedu Walker,IND,103,0.04%


In [15]:
df.isnull().sum()

Year          0
Old Region    0
New Region    0
Code          0
Candidate     0
Party         0
Votes         0
Votes(%)      0
dtype: int64

In [16]:
df.dtypes

Year          int64
Old Region      str
New Region      str
Code            str
Candidate       str
Party           str
Votes         int64
Votes(%)        str
dtype: object

In [17]:
# remove % sign and convert to number
df['Votes(%)'] = df['Votes(%)'].str.replace('%', '').str.strip()
df['Votes(%)'] = pd.to_numeric(df['Votes(%)'], errors='coerce')

# make Votes numeric
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

# remove hidden non-breaking space in region names
df['New Region'] = df['New Region'].str.replace('\xa0', ' ', regex=False).str.strip()

# fill any missing values with 0 (imputation)
df['Votes'] = df['Votes'].fillna(0).astype(int)
df['Votes(%)'] = df['Votes(%)'].fillna(0.0)

print('Done. Shape:', df.shape)
df.head()

Done. Shape: (615, 8)


,Year,Old Region,New Region,Code,Candidate,Party,Votes,Votes(%)
0,2020,Brong Ahafo Region,Ahafo Region,NPP,Nana Akufo Addo,NPP,145584,55.04
1,2020,Brong Ahafo Region,Ahafo Region,NDC,John Dramani Mahama,NDC,116485,44.04
2,2020,Brong Ahafo Region,Ahafo Region,Others,Ivor Kobina Greenstreet,CPP,191,0.07
3,2020,Brong Ahafo Region,Ahafo Region,Others,David Asibi Ayindenaba Apasera,PNC,83,0.03
4,2020,Brong Ahafo Region,Ahafo Region,Others,Alfred Kwame Asiedu Walker,IND,103,0.04


## 2. Load and Clean the PDF

In [18]:
doc = fitz.open('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
total_pages = doc.page_count
print('Total pages:', total_pages)

all_text = ''
for page in doc:
    text = page.get_text()
    if len(text.strip()) < 50:   # skip blank pages
        continue
    # remove repeated header / footer text
    text = text.replace('Resetting the Economy for the Ghana We Want', '')
    text = text.replace('2025 Budget', '')
    # drop special symbols that cause encoding errors
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    all_text += text

doc.close()
print('Total words:', len(all_text.split()))

Total pages: 252
Total words: 94171


In [19]:
print(all_text[1000:1500])

lectronic copies can be downloaded from our website (www.mofep.gov.gh)  
 
 
 
i 
 
TABLE OF CONTENTS 
List of Tables .................................................................................................................. iii 
List of Figures ................................................................................................................. v 
Acronyms and Abbreviations .......................................................................................... vi 
SECTION


## 3. Chunking

A chunk is a small piece of text. I split the documents into chunks so the chatbot can search through them.

**Overlap** means repeating a few words at the boundary so we do not cut a sentence in half.

I compared two strategies:
- Small chunks: 100 words, 20 word overlap
- Large chunks: 300 words, 50 word overlap

I chose **large chunks (300 words)** for the PDF because budget paragraphs need more context to make sense. For the CSV, each row is already one complete fact so 1 row = 1 chunk.

In [20]:
# CSV - 1 row = 1 chunk
csv_chunks = []
for i, row in df.iterrows():
    text = (f"In the {row['Year']} Ghana election, {row['Candidate']} "
            f"from {row['Party']} got {row['Votes']:,} votes ({row['Votes(%)']}%) "
            f"in {row['New Region']}.")
    csv_chunks.append({'id': f'csv_{i}', 'source': 'election', 'text': text})

print('CSV chunks:', len(csv_chunks))
print('Example:', csv_chunks[0]['text'])

CSV chunks: 615
Example: In the 2020 Ghana election, Nana Akufo Addo from NPP got 145,584 votes (55.04%) in Ahafo Region.


In [21]:
# PDF - sliding window chunking
def chunk_text(text, size, overlap):
    words = text.split()
    step = size - overlap
    chunks = []
    for i, start in enumerate(range(0, len(words), step)):
        piece = words[start : start + size]
        if len(piece) < 30:
            break
        chunks.append({'id': f'pdf_{i}', 'source': 'budget', 'text': ' '.join(piece)})
    return chunks

small_chunks = chunk_text(all_text, size=100, overlap=20)
large_chunks = chunk_text(all_text, size=300, overlap=50)

print('Small chunks (100 words):', len(small_chunks))
print('Large chunks (300 words):', len(large_chunks))

Small chunks (100 words): 1177
Large chunks (300 words): 377


In [22]:
# compare both with a test query using keyword matching
query_words = set('education spending Ghana 2025'.lower().split())

def best_match(q, chunks):
    best = max(chunks, key=lambda c: len(q & set(c['text'].lower().split())))
    score = len(q & set(best['text'].lower().split()))
    return score, best['text'][:200]

s1, _ = best_match(query_words, small_chunks)
s2, preview = best_match(query_words, large_chunks)

print('Small chunk score:', s1)
print('Large chunk score:', s2)
print()
print('Best large chunk preview:')
print(preview)

Small chunk score: 3
Large chunk score: 3

Best large chunk preview:
(2019-2024) ................ 31 Figure 4: Evolution of Ghana’s Long-Term Local Currency Rating by Fitch, S&P, & Moody’s (2003-2024) ....................................................................


## 4. Save Chunks for Part B

In [23]:
os.makedirs('chunks', exist_ok=True)

with open('chunks/csv_chunks.json', 'w') as f:
    json.dump(csv_chunks, f)

with open('chunks/pdf_chunks.json', 'w') as f:
    json.dump(large_chunks, f)

print('csv chunks saved:', len(csv_chunks))
print('pdf chunks saved:', len(large_chunks))

csv chunks saved: 615
pdf chunks saved: 377
